## Part 1: Imports and Setup

Import the required libraries for this project.

In [1]:
# Import standard library modules needed throughout the project
import csv          # for reading the CSV dataset
import heapq        # for heap-based top-K operations
import re           # for regex-based suspicious pattern matching

from collections import deque          # efficient O(1) queue for BFS
from typing import Dict, List, Set     # type hints for clarity

# Third-party visualisation libraries (install with: pip install networkx matplotlib scipy)
import networkx as nx
import matplotlib.pyplot as plt

## Part 2: Data Loading

Implement a function to load CSV data into a list of dictionaries.

In [2]:
def load_data(file_path: str) -> List[Dict[str, str]]:
    """Load transaction records from a CSV file.

    Parameters
    ----------
    file_path : str
        Path to the CSV file to read.

    Returns
    -------
    List[Dict[str, str]]
        Each element is one row from the CSV, keyed by column header.
    """
    transactions: List[Dict[str, str]] = []

    # Open the file with UTF-8 encoding; newline='' prevents double line-endings
    with open(file_path, newline='', encoding='utf-8') as csv_file:
        # DictReader automatically uses the first row as field names
        reader = csv.DictReader(csv_file)
        for row in reader:
            transactions.append(dict(row))   # convert OrderedDict -> plain dict

    return transactions

## Part 3: Graph Construction

Build a graph where users are connected to products they purchased.

In [3]:
def build_graph(data: List[Dict[str, str]]) -> Dict[str, Set[str]]:
    """Build an undirected bipartite adjacency-list graph from transaction records.

    Each user node is connected to the product nodes it purchased, and each
    product node links back to the users that bought it.  Storing edges in
    both directions lets BFS traverse the full connected component from any
    starting node (user *or* product).

    Parameters
    ----------
    data : List[Dict[str, str]]
        Loaded transaction records (output of load_data).

    Returns
    -------
    Dict[str, Set[str]]
        Adjacency list: node -> set of neighbouring nodes.
    """
    graph: Dict[str, Set[str]] = {}

    for transaction in data:
        user_id    = transaction['user_id']
        product_id = transaction['product_id']

        # Add product to user's neighbour set
        graph.setdefault(user_id, set()).add(product_id)

        # Add user to product's neighbour set (undirected edge -- required for BFS)
        graph.setdefault(product_id, set()).add(user_id)

    return graph

## Part 4: Graph Traversal (BFS)

Implement Breadth-First Search to traverse the graph.

In [4]:
def bfs(graph: Dict[str, Set[str]], start: str) -> Set[str]:
    """Breadth-First Search from a starting node.

    Explores the graph layer by layer using a FIFO queue (deque).
    Time complexity: O(V + E) where V = nodes visited, E = edges traversed.

    Parameters
    ----------
    graph : Dict[str, Set[str]]
        Adjacency list representation of the graph.
    start : str
        Node from which to begin traversal.

    Returns
    -------
    Set[str]
        All nodes reachable from 'start' (including start itself).
    """
    visited: Set[str] = set()
    queue: deque = deque([start])   # initialise the queue with the start node

    while queue:
        node = queue.popleft()      # dequeue the front node -- O(1) with deque

        if node not in visited:
            visited.add(node)       # mark as visited so we don't revisit it

            # Enqueue all unvisited neighbours
            for neighbor in graph.get(node, set()):
                if neighbor not in visited:
                    queue.append(neighbor)

    return visited

## Part 5: Pattern Matching (Regex)

Find suspicious transactions by searching for suspicious keywords.

In [5]:
def find_suspicious(data: List[Dict[str, str]], keywords: List[str]) -> List[Dict[str, str]]:
    """Return transactions whose description field contains any suspicious keyword.

    The keywords are joined into a single alternation pattern (kw1|kw2|...)
    which is compiled once for efficiency, then matched case-insensitively
    against each transaction's 'description' field.

    Time complexity: O(N * L) where N = number of transactions,
                                      L = average description length.

    Parameters
    ----------
    data     : List[Dict[str, str]]  Transaction records to search.
    keywords : List[str]             Suspicious words (e.g. ['urgent','lottery','refund']).

    Returns
    -------
    List[Dict[str, str]]
        Subset of 'data' whose description matched at least one keyword.
    """
    # Build one compiled pattern from all keywords (avoids re-compiling per row)
    pattern = re.compile('|'.join(keywords), re.IGNORECASE)

    return [txn for txn in data if pattern.search(txn['description'])]

## Part 6: Top-K Products (Heap)

Use a heap to efficiently find the top-K most purchased products.

In [6]:
def top_k_products(data: List[Dict[str, str]], k: int) -> List[str]:
    """Return the k most frequently purchased product IDs.

    Counts purchases with a hash map (O(N)), then uses heapq.nlargest to
    extract the top-k entries in O(N log k) -- much faster than a full sort
    when k << N.

    Parameters
    ----------
    data : List[Dict[str, str]]  Transaction records.
    k    : int                   Number of top products to return.

    Returns
    -------
    List[str]
        Product IDs sorted from most to least purchased (length <= k).
    """
    # Count how many times each product appears across all transactions
    freq: Dict[str, int] = {}
    for txn in data:
        pid = txn['product_id']
        freq[pid] = freq.get(pid, 0) + 1

    # heapq.nlargest selects the top-k items by frequency in O(N log k)
    top_items = heapq.nlargest(k, freq.items(), key=lambda item: item[1])

    # Return only the product IDs, not the raw counts
    return [product for product, _count in top_items]

## Part 7: Load Data

Load the dataset for analysis.

In [ ]:
# Load all transaction records from the notional dataset CSV file
dataset = load_data('notional_dataset.csv')

# Quick sanity check -- confirm data loaded and inspect one record
print(f"Loaded {len(dataset)} transactions.")
print("\nSample transaction (first row):")
print(dataset[0])

## Part 8: Exploratory Data Analysis

Analyze the dataset to understand its structure.

In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

# 1. Total number of transactions
total_transactions = len(dataset)
print(f"\nTotal transactions: {total_transactions}")

# 2. Sample transaction (already printed above, repeated here for context)
print(f"\nSample transaction:\n  {dataset[0]}")

# 3. Count unique users, products, and description types
# Set comprehensions deduplicate IDs in a single O(N) pass
unique_users        = {txn['user_id']     for txn in dataset}
unique_products     = {txn['product_id']  for txn in dataset}
unique_descriptions = {txn['description'] for txn in dataset}

print(f"\nUnique users:        {len(unique_users)}")
print(f"Unique products:     {len(unique_products)}")
print(f"Unique descriptions: {len(unique_descriptions)}")

# 4. Transaction type distribution
desc_counts: Dict[str, int] = {}
for txn in dataset:
    desc = txn['description']
    desc_counts[desc] = desc_counts.get(desc, 0) + 1

print("\nTransaction type distribution:")
# Sort by count descending so the most common types appear first
for desc, count in sorted(desc_counts.items(), key=lambda x: -x[1]):
    pct = count / total_transactions * 100
    print(f"  {desc:<20} {count:>6}  ({pct:.1f}%)")

# 5. Amount statistics -- stored as strings in CSV so we cast to float
amounts = [float(txn['amount']) for txn in dataset]

print(f"\nAmount statistics:")
print(f"  Min:     ${min(amounts):>10.2f}")
print(f"  Max:     ${max(amounts):>10.2f}")
print(f"  Average: ${sum(amounts) / len(amounts):>10.2f}")

# 6 & 7. Suspicious transaction count and rate
# Using the three keywords specified in the project description
suspicious_keywords = ["urgent", "lottery", "refund"]

suspicious_txns  = find_suspicious(dataset, suspicious_keywords)
suspicious_count = len(suspicious_txns)
suspicious_rate  = suspicious_count / total_transactions * 100

print(f"\nSuspicious transactions: {suspicious_count} ({suspicious_rate:.2f}%)")

## Part 9: Build and Display Graph

Create the user-product graph and print its structure.

In [ ]:
# Build the adjacency-list graph from the loaded dataset.
# Each user node connects to every product it purchased (and vice-versa)
# so that BFS can traverse the full component from any starting node.
graph = build_graph(dataset)

# Print a small sample rather than the entire graph to keep output readable
print("User-Product Graph (first 5 nodes shown):")
for node, neighbors in list(graph.items())[:5]:
    print(f"  {node}: {neighbors}")

print(f"\n  ... ({len(graph)} total nodes across users and products)")

## Part 10: Visualize Graph with NetworkX

Create a visual representation of the user-product network.

In [ ]:
# Visualize the graph using NetworkX
# Actions:
# 1. Create a NetworkX Graph object
# 2. Add edges from the user-product graph
# 3. Separate nodes into users and products (using node name prefixes)
# 4. Use spring_layout for positioning
# 5. Draw users in lightblue and products in lightcoral
# 6. Print graph statistics (nodes, edges, users, products, average degree)

import scipy   # required by some NetworkX layout algorithms

# Build a NetworkX undirected graph from our adjacency-list dictionary
G = nx.Graph()

# Add one edge for every (user, product) connection stored in the graph dict
for user, neighbors in graph.items():
    for product in neighbors:
        G.add_edge(user, product)

# Create the visualisation canvas
plt.figure(figsize=(20, 16))

# Spring layout distributes nodes to minimise edge crossings
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)

# Colour-code nodes by type: users start with 'U', products start with 'P'
users    = [node for node in G.nodes() if node.startswith('U')]
products = [node for node in G.nodes() if node.startswith('P')]

nx.draw_networkx_nodes(G, pos, nodelist=users,    node_color='lightblue',
                       node_size=100, label='Users',    alpha=0.7)
nx.draw_networkx_nodes(G, pos, nodelist=products, node_color='lightcoral',
                       node_size=100, label='Products', alpha=0.7)
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5)

plt.title('User-Product Transaction Graph', fontsize=16, fontweight='bold')
plt.legend(scatterpoints=1, fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

# Print graph statistics
print(f"\nGraph Statistics:")
print(f"  Total nodes:        {G.number_of_nodes()}")
print(f"  Total edges:        {G.number_of_edges()}")
print(f"  Number of users:    {len(users)}")
print(f"  Number of products: {len(products)}")
avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()
print(f"  Average degree: {avg_degree:.2f}")

## Part 11: Connected Components Detection

Find all connected components in the graph.

In [ ]:
def detect_connected_components(graph: Dict[str, Set[str]]) -> List[Set[str]]:
    """Find every connected component in the graph using BFS.

    A connected component is a maximal set of nodes where every node can
    reach every other node.  We iterate over all graph nodes and call bfs()
    on each unvisited one; when BFS finishes it has discovered one full
    component.

    Time complexity: O(V + E) -- each node and edge is visited exactly once.

    Parameters
    ----------
    graph : Dict[str, Set[str]]
        Adjacency list (output of build_graph).

    Returns
    -------
    List[Set[str]]
        List of sets, where each set contains the nodes of one component.
    """
    visited: Set[str] = set()       # nodes already assigned to a component
    components: List[Set[str]] = []

    for node in graph:
        if node not in visited:
            # BFS discovers every node reachable from this unvisited node
            component = bfs(graph, node)
            components.append(component)
            # Mark the entire component as visited to skip it in future iterations
            visited.update(component)

    return components


# ── Analyse results ──────────────────────────────────────────────────────────
components = detect_connected_components(graph)

print(f"Total connected components: {len(components)}")

# Sort by size descending so the most connected clusters appear first
components_sorted = sorted(components, key=len, reverse=True)

print("\nTop 10 components by size:")
for i, comp in enumerate(components_sorted[:10], 1):
    comp_users    = [n for n in comp if n.startswith('U')]
    comp_products = [n for n in comp if n.startswith('P')]
    print(f"  Component {i:>2}: {len(comp):>5} nodes  "
          f"({len(comp_users)} users, {len(comp_products)} products)")

## Part 12: Suspicious Cluster Detection

Identify high-risk clusters of users based on suspicious transaction rates.

In [ ]:
print("=" * 50)
print("SUSPICIOUS CLUSTER DETECTION")
print("=" * 50)

# Reuse the keywords defined in the EDA section
suspicious_keywords = ["urgent", "lottery", "refund"]

# Pre-build a user -> [transactions] lookup so component analysis is O(users)
# rather than O(users * N) if we scanned the full dataset each time.
user_transactions: Dict[str, List[Dict[str, str]]] = {}
for txn in dataset:
    uid = txn['user_id']
    user_transactions.setdefault(uid, []).append(txn)

# Examine the top 500 components (by size) for suspicious activity
top_components = sorted(components, key=len, reverse=True)[:500]

# Track product frequencies across all detected high-risk clusters
suspicious_product_freq: Dict[str, int] = {}
high_risk_count = 0

for idx, component in enumerate(top_components):
    # Separate component nodes into users (prefix 'U') and products (prefix 'P')
    comp_users    = {node for node in component if node.startswith('U')}
    comp_products = {node for node in component if node.startswith('P')}

    # Gather every transaction made by a user who belongs to this component
    comp_transactions: List[Dict[str, str]] = []
    for user in comp_users:
        comp_transactions.extend(user_transactions.get(user, []))

    total = len(comp_transactions)
    if total == 0:
        continue   # skip empty components

    # Count suspicious transactions within this component
    comp_suspicious  = find_suspicious(comp_transactions, suspicious_keywords)
    suspicious_count = len(comp_suspicious)
    rate             = suspicious_count / total * 100

    # Classify the component as HIGH RISK if its suspicious rate exceeds 20%
    if rate > 20:
        high_risk_count += 1
        print(f"\nHIGH RISK CLUSTER #{high_risk_count}:")
        print(f"  Nodes       : {len(component)} ({len(comp_users)} users, {len(comp_products)} products)")
        print(f"  Transactions: {total} total | {suspicious_count} suspicious ({rate:.1f}%)")

        # Record which products appear in the suspicious transactions of this cluster
        for txn in comp_suspicious:
            pid = txn['product_id']
            suspicious_product_freq[pid] = suspicious_product_freq.get(pid, 0) + 1

# Use heapq.nlargest to surface the top 10 most-flagged products
top_suspicious_products = heapq.nlargest(
    10, suspicious_product_freq.items(), key=lambda item: item[1]
)

print("\n" + "=" * 50)
print("TOP 10 PRODUCTS IN HIGH-RISK CLUSTERS")
print("=" * 50)

if top_suspicious_products:
    for rank, (product, count) in enumerate(top_suspicious_products, 1):
        print(f"  {rank:>2}. {product}: {count} suspicious transaction(s)")
else:
    print("  No high-risk clusters detected with current 20% threshold.")

## Part 13: Top product recommendation

In [ ]:
# 1. Find top 10 products overall using the heap-based top_k_products function
suspicious      = find_suspicious(dataset, suspicious_keywords)
recommendations = top_k_products(dataset, 10)

print(f"Total suspicious transactions: {len(suspicious)}")
print(f"\nTop 10 products (overall frequency):")
for rank, pid in enumerate(recommendations, 1):
    print(f"  {rank:>2}. {pid}")

# 2. Compare against the top products found inside high-risk clusters (from Part 12)
top_suspicious_product_ids = [p for p, _ in top_suspicious_products]

print(f"\nTop 10 products (in high-risk clusters):")
for rank, pid in enumerate(top_suspicious_product_ids, 1):
    print(f"  {rank:>2}. {pid}")

# Products appearing in BOTH lists are popular overall AND concentrated in risk zones
overlap = set(recommendations) & set(top_suspicious_product_ids)
print(f"\nOverlap between both lists ({len(overlap)} product(s)): {sorted(overlap)}")
if overlap:
    print("  => These products are both popular overall and appear frequently")
    print("     in high-risk clusters -- they may warrant further investigation.")
else:
    print("  => No overlap: the most popular products are NOT concentrated in")
    print("     high-risk clusters, suggesting fraud is product-agnostic here.")

## Reflection Questions

1. Why might the top 10 products overall be different from the top 10 products in suspicious clusters?
2. How does using a graph structure help in detecting fraud?
3. What is the time complexity of your BFS implementation?
4. How would you modify your solution to detect more sophisticated fraud patterns?
5. What are the limitations of using regex for fraud detection?
6. Use complexity analysis to find the complexity of your project and discuss trade-offs that you could do to decrease the complexity. You do not have to implement the suggestions. 